In [1]:
import os
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
import getpass
load_dotenv(".env.RAG")

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embd = GoogleGenerativeAIEmbeddings(model="models/gemini-2.5-flash")






e:\Projects\RAG\LangRAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# ! pip install -U langchain-huggingface

In [3]:

# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5359.63it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# file_path = "./Data/Res.pdf"
folder_path="./Data"

loader = DirectoryLoader(
    folder_path,
    glob="**/*",


)


docs = loader.load()
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Md.Ghalib Faruqe

Adresse : Matikata, Dhaka Cantonment, Dhaka Phone : 01719814500 Email : ghalib.arnob@gmail.com LinkedIn : linkedin.com/in/ghalibfaruqe

American International University – Bangladesh

{'source': 'Data\\Res.pdf'}


- used clear_func to formate the text for better embeding

In [5]:

import re
def cleaner_func(docs):

  if isinstance(docs, list):
    cleared=[]
    for doc in docs:
      if isinstance(doc, str):
         doc = re.sub(r'\n+', '\n', doc)
         doc = re.sub(r'\s+', ' ', doc)
         cleared.append(doc.strip())
      else:
        cleared.append(doc)
    return cleared

    

  elif isinstance(docs, str):
    doc= ''.join(str(d)for d in docs)

  docs = re.sub(r'\n+', '\n', docs)
  docs = re.sub(r'\s+', ' ', docs)
  return docs.strip()

cleaned_docs = cleaner_func(docs)

In [6]:
text_split = RecursiveCharacterTextSplitter(
    chunk_size= 400, 
    chunk_overlap=100, 
    length_function=len,
    add_start_index=True

)
cleaned_chunk = text_split.split_documents(cleaned_docs)
print(len(cleaned_chunk))

120


In [7]:
from langchain_chroma import Chroma
vector_store = Chroma.from_documents(
    documents=cleaned_chunk,
    collection_name="Lang_Rag_hf",
    embedding=embd,
    persist_directory="./Rag/chroma_DB",
    
)


In [8]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":7}
)



In [9]:
query =  "Which university is mentioned in the resume education section?"

In [10]:
# ! pip install -U huggingface_hub

In [11]:
hf_token=os.getenv("HF_TOKEN")

In [12]:
# ! pip install rank_bm25

In [15]:
from rank_bm25 import BM25Okapi
text_chunks = [chunk.page_content for chunk in cleaned_chunk]
tokenized_chunks = [chunk.split() for chunk in text_chunks]
bm25 = BM25Okapi(tokenized_chunks)
tokenize_query= query.split()
bm25_result = bm25.get_top_n(tokenize_query,tokenized_chunks,n=5)

In [ ]:
for i, result in enumerate(bm25_result,1):
    print(f"{i}.. {result[:100]}")

In [17]:

vector_output = retriever.invoke(query)
vector_texts = [chunk.page_content for chunk in vector_output]
combined_output = bm25_result+vector_texts


In [ ]:
seen = set()
result_output = []
for item in combined_output:
    if item not in seen:
        seen.add(item)
        result_output.append(item)

print(f"BM25 results: {len(bm25_results)}")
print(f"Vector results: {len(vector_results)}")
print(f"Combined unique results: {len(result_output)}")

for i, result in enumerate(result_output, 1):
    print(f"\nResult {i}")
    print(result[:200] + "..." if len(result) > 200 else result)

In [23]:
# Debug: Check what you're actually working with
print("=== DEBUGGING ===")
print(f"Type of bm25_result: {type(bm25_result)}")
if bm25_result and len(bm25_result) > 0:
    print(f"Type of first item in bm25_result: {type(bm25_result[0])}")
    if isinstance(bm25_result[0], list):
        print(f"First item preview: {bm25_result[0][:5]}...")
    else:
        print(f"First item preview: {bm25_result[0][:100]}...")

print(f"\nType of vector_texts: {type(vector_texts)}")
if vector_texts and len(vector_texts) > 0:
    print(f"Type of first item in vector_texts: {type(vector_texts[0])}")
    print(f"First item preview: {vector_texts[0][:100]}...")

print(f"\nType of combined_output: {type(combined_output)}")
if combined_output and len(combined_output) > 0:
    print(f"Type of first item in combined_output: {type(combined_output[0])}")

# Fix based on what you see
if bm25_result and isinstance(bm25_result[0], list):
    print("\nConverting BM25 results from lists to strings...")
    bm25_results = [' '.join(tokens) for tokens in bm25_result]
else:
    bm25_results = bm25_result

if vector_texts and isinstance(vector_texts[0], list):
    print("Converting vector results from lists to strings...")
    vector_results = [' '.join(tokens) for tokens in vector_texts]
else:
    vector_results = vector_texts

# Now combine
combined = bm25_results + vector_results
seen = set()
unique_results = []
for item in combined:
    if item not in seen:
        seen.add(item)
        unique_results.append(item)

print(f"\nFinal unique results: {len(unique_results)}")

=== DEBUGGING ===
Type of bm25_result: <class 'list'>
Type of first item in bm25_result: <class 'list'>
First item preview: ['CAREER', 'OBJECTIVE', 'Dedicated', 'education', 'enthusiast']...

Type of vector_texts: <class 'list'>
Type of first item in vector_texts: <class 'str'>
First item preview: CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical applic...

Type of combined_output: <class 'list'>
Type of first item in combined_output: <class 'list'>

Converting BM25 results from lists to strings...

Final unique results: 6


In [24]:
if bm25_result and isinstance(bm25_result[0], list):
    bm25_results = [' '.join(tokens) for tokens in bm25_result]
else:
    bm25_results = bm25_result

chunk_scores = {}

for i, chunk in enumerate(bm25_results):
    chunk_scores[chunk] = chunk_scores.get(chunk, 0) + (5 - i)

for i, chunk in enumerate(vector_texts):
    chunk_scores[chunk] = chunk_scores.get(chunk, 0) + (7 - i)

sorted_chunks = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nTotal unique chunks: {len(sorted_chunks)}")


for i, (chunk, score) in enumerate(sorted_chunks[:5]):
    print(f"\n  Rank {i+1} (Score: {score})  ")
    print(chunk[:200] )


Total unique chunks: 6

  Rank 1 (Score: 32)  
CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The George Washington University to pursue a Bachelor of 

  Rank 2 (Score: 4)  
Led a campus-wide recycling program, increasing recycled items submitted by 31% within the ﬁrst six months. Used Canva to design banners, images, and posts for the club, resulting in a 44% increase in

  Rank 3 (Score: 3)  
Coordinated 2 science education outreach programs, reaching 2,000+ high school students and increasing engagement by 19% Extensively researched the latest education strategies and introduced 4 innovat

  Rank 4 (Score: 2)  
various club events to promote sustainable practices among university students, improving community involvement. Organized 7 educational seminars on climate change awareness, attended by 211 students 

  Rank 5 (Score: 1)  
assessments, which helped clients lose an average of 8% 

In [ ]:
# ! pip install sentence-transformers

In [20]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6512.41it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
pairs=[[query, chunk] for chunk in combined_output]
score = reranker.predict(pairs)
ranked_chunks=[chunk for _, chunk in sorted(zip(score, combined_output),
reverse=True)]

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np

def rerank_results(query, chunks, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2'):
    """
    Rerank chunks based on relevance to query
    """
    reranker = CrossEncoder(model_name)
    
    clean_chunks = []
    for chunk in chunks:
        if isinstance(chunk, str):
            clean_chunks.append(chunk)
        elif isinstance(chunk, list):
            clean_chunks.append(' '.join(chunk))
        elif hasattr(chunk, 'page_content'):  
            clean_chunks.append(chunk.page_content)
        else:
            clean_chunks.append(str(chunk))
    
    pairs = [[query, chunk] for chunk in clean_chunks]
    
    try:
        scores = reranker.predict(pairs)
    except Exception as e:
        print(f"Error during prediction: {e}")
        return chunks, [0] * len(chunks)
    
    ranked_pairs = sorted(zip(scores, clean_chunks), reverse=True)
    ranked_chunks = [chunk for score, chunk in ranked_pairs]
    ranked_scores = [score for score, chunk in ranked_pairs]
    
    return ranked_chunks, ranked_scores


ranked_chunks, scores = rerank_results(query, combined_output)

print(f"Original chunks: {len(combined_output)}")
print(f"Ranked chunks: {len(ranked_chunks)}")

for i, (chunk, score) in enumerate(zip(ranked_chunks[:5], scores[:5]), 1):
    print(f"\nRank {i} (Score: {score:.4f})")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6639.86it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original chunks: 12
Ranked chunks: 12

Rank 1 (Score: -3.7121)
CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The Geo...

Rank 2 (Score: -3.7121)
CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The Geo...

Rank 3 (Score: -3.7121)
CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The Geo...

Rank 4 (Score: -3.7121)
CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The Geo...

Rank 5 (Score: -3.7121)
CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The Geo...


In [27]:
top_chunk= ranked_chunks[:3]


In [36]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [29]:
pipline= pipeline(
    "text-generation",
    model="microsoft/phi-2",
    max_new_tokens=512,
    temperature=0.3
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]e:\Projects\RAG\LangRAG\RAG\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ghali\.cache\huggingface\hub\models--microsoft--phi-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 453/453 [00:00<00:00, 4296.19it/s]
P

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [39]:
print(top_chunk)

['CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The George Washington University to pursue a Bachelor of Arts Education. With a ﬁrm belief in the transformative power of education and a profound commitment to innovative teaching methods, I am eager to reﬁne my skilset to be able to shape the future of', 'CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The George Washington University to pursue a Bachelor of Arts Education. With a ﬁrm belief in the transformative power of education and a profound commitment to innovative teaching methods, I am eager to reﬁne my skilset to be able to shape the future of', 'CAREER OBJECTIVE Dedicated education enthusiast yearning to deepen my knowledge and practical application in education, seeking to transfer to The George Washington University to pursue a Bachelor

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"

text_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,  
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
    length_function=len,
)
cleaned_chunk = text_split.split_documents(cleaned_docs)

text_chunks = [chunk.page_content for chunk in cleaned_chunk]
tokenized_chunks = [chunk.split() for chunk in text_chunks]
bm25 = BM25Okapi(tokenized_chunks)

def retrieve(query, top_k=5):
    vector_docs = retriever.invoke(query)
    vector_texts = [doc.page_content for doc in vector_docs]
    
    tokenized_query = query.split()
    bm25_texts = bm25.get_top_n(tokenized_query, text_chunks, n=top_k)
    
    all_texts = []
    seen = set()
    for text in bm25_texts + vector_texts:
        if text not in seen:
            seen.add(text)
            all_texts.append(text)
    
    return all_texts[:top_k*2]

def rerank(query, texts):
    if 'reranker' not in globals():
        return texts[:3]
    
    pairs = [[query, text] for text in texts]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, texts), reverse=True)
    return [text for score, text in ranked[:3]]

def ask(question):
    
    texts = retrieve(question)
    
    top_texts = rerank(question, texts)
    
    if 'qa_chain' in globals():
        context = "\n\n".join(top_texts)
        response = qa_chain.run(context=context, question=question)
        if "Answer:" in response:
            response = response.split("Answer:")[-1].strip()
        return response
    else:
        return f"Found {len(top_texts)} relevant passages."

print(ask("Which university does Hunter Jacobson studies mentioned in the resume? He is a student of human resource"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Hunter Jacobson studies at the University of Colorado.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
